In [1]:
# 1. Setup & Configuration
import pandas as pd
import time
from institution_checker import run_pipeline, set_api_key
from name_matcher.pipeline import NameMatcher, NameMatcherConfig

# Configure API Key
set_api_key("sk-3f1dbbf2450e46ab9541dffba4f18ec6")

# Configure Name Matching
config = NameMatcherConfig(name_column="name", use_tqdm=True)
config.llm_token = "sk-3f1dbbf2450e46ab9541dffba4f18ec6"

print("✅ System initialized.")

C:\Users\setiawa\Documents\institution_checker\src\institution_checker\llm_processor.py:100: FutureWarning: Possible set difference at position 14
  DATE_RANGE_PATTERN = re.compile(r'\b(\d{4})\s*[---]\s*(\d{4})\b')


✅ System initialized.


## 2. Data Sampling (100 Names)
We select a mix of known Purdue laureates (VIPs) and random awardees to demonstrate accuracy.

In [2]:
# Load Source Data
source_path = r"E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx"
try:
    df = pd.read_excel(source_path, dtype=str)
except FileNotFoundError:
    # Fallback if E: drive is not available
    print("⚠️ Source file not found. Generating synthetic data for demo.")
    df = pd.DataFrame({'name': [
        "John B. Fenn", "Ei-ichi Negishi", "Herbert C. Brown", "Akira Suzuki", 
        "Vernon L. Smith", "Ben R. Mottelson", "E. M. Purcell", "Julian Schwinger", 
        "Wolfgang Pauli"
    ] + [f"Random Person {i}" for i in range(91)]})

# Define VIPs (Known Purdue Connections)
vip_names = [
    "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
    "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
    "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
]

# 1. Select VIPs
vip_mask = df['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
df_vips = df[vip_mask].copy()

# 2. Select Random Sample (to reach 100 total)
df_others = df[~vip_mask].copy()
remaining_count = 100 - len(df_vips)
df_random = df_others.sample(n=min(remaining_count, len(df_others)), random_state=42)

# 3. Combine
df_demo = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)

print(f"📊 Demo Dataset Prepared:")
print(f"   - VIPs (Known): {len(df_vips)}")
print(f"   - Random Sample: {len(df_random)}")
print(f"   - Total: {len(df_demo)} names")
display(df_demo.head())

📊 Demo Dataset Prepared:
   - VIPs (Known): 24
   - Random Sample: 76
   - Total: 76 names


,in AcA,Purdue,mexclusive_conn,checked (blue),id,award,name,year,affiliation,govid,...,year elected,state,previous service,current service,discipline,related_links,note,position,subaward,fulltext
0,NaN,True,Postdoc,1,2230,Nobel Prize-Chemistry,Akira Suzuki,2010,NaN,248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,False,True,Alumni,1,819,American Philosophical Society Membership,Ben R. Mottelson,2011,"The Niels Bohr Institute, University of Copen...",91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Visiting Professor,NaN,NaN
2,False,True,Alumni,1,2023,NAS Members,E. M. Purcell,1951,Harvard University,222,...,Member (elected 1951),NaN,NaN,NaN,Scientific Discipline: Physics,http://www.nasonline.org/publications/biograph...,NaN,NaN,NaN,NaN
3,False,True,Faculty,1,101,American Academy of Arts and Sciences Member/F...,Ei-ichi Negishi,2011,NaN,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,True,Faculty,1,404,Priestley Medal,Herbert C. Brown,1981,Purdue University,41,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Professor Emeritus,NaN,NaN


## 3. Entity Resolution
Clustering similar names to ensure we don't process duplicates (e.g., "John Fenn" vs "J.B. Fenn").

In [3]:
print("🧩 Clustering names...")
matcher = NameMatcher(config)
result = matcher.cluster(df_demo)
df_clustered = result.dataframe

# Select Representatives
cluster_reps = df_clustered.groupby('cluster_label')['name'].first().reset_index()
cluster_reps.columns = ['cluster_label', 'representative_name']

# Ensure VIPs are the representatives for their clusters (for better search results)
for vip in vip_names:
    vip_rows = df_clustered[df_clustered['name'].str.contains(vip, case=False, na=False)]
    if not vip_rows.empty:
        cluster_label = vip_rows.iloc[0]['cluster_label']
        cluster_reps.loc[cluster_reps['cluster_label'] == cluster_label, 'representative_name'] = vip

names_to_check = cluster_reps['representative_name'].tolist()
print(f"✅ Reduced to {len(names_to_check)} unique clusters to verify.")

🧩 Clustering names...
--- Name Matcher Process Started (v6.8.8 Improved Canonical Name Selection) ---

1. Loading and preparing data...
   Loaded 76 names. Done in 0.00s
2. Vectorizing names for similarity scoring...
   Done in 0.00s
3. Generating candidate pairs via Blocking...
   Generated 4 candidate pairs from blocking.
   Done in 0.00s
4. Applying Stricter Cluster-Centric Filtering...


   Filtering Pairs: 100%|██████████| 4/4 [00:00<?, ?pair/s]

   Filter Stats: {'merged_expansion': 4}
   Done in 0.00s
5. Reviewing ambiguous pairs with LLM...
   Done in 0.00s
6. Finalizing cluster data structures...
   Done in 0.00s
7. Refining clusters by ejecting outliers...
   Found and ejected 0 outliers from clusters.
   Done in 0.00s
8. Assigning canonical names and saving results...

--- Results Summary ---
   - Total names processed: 76
   - Unique entities (clusters) found: 72

   --- Sample of Largest Clusters Found ---
   Cluster 1 (Size: 2): 'Edward Mills Purcell'
     - E. M. Purcell
     - Edward Mills Purcell
   Cluster 2 (Size: 2): 'Julian Seymour Schwinger'
     - Julian Schwinger
     - Julian Seymour Schwinger
   Cluster 3 (Size: 2): 'Harlan Fiske Stone'
     - Harlan F. Stone
     - Harlan Fiske Stone
   Cluster 4 (Size: 2): 'Armin Dale Kaiser'
     - A. Dale Kaiser
     - Armin Dale Kaiser
   Done in 0.00s

--- Name Matcher Process Finished in 0.02 seconds ---
✅ Reduced to 72 unique clusters to verify.


## 4. Verification Pipeline
Running the AI-driven search and verification process.

In [4]:
print(f"🚀 Starting verification for {len(names_to_check)} names...")
start_time = time.time()

# Run the pipeline
results = await run_pipeline(
    names_to_check, 
    batch_size=10,              # Efficient batch size
    use_enhanced_search=True,   # Deep search for accuracy
    dataset_profile="high_connection", # Conservative filtering
    use_dynamic_batching=True
)

elapsed = time.time() - start_time
print(f"🏁 Completed in {elapsed:.1f}s ({elapsed/len(names_to_check):.2f}s per name)")

🚀 Starting verification for 72 names...
[PIPELINE] Starting DYNAMIC BATCHING mode
[PIPELINE] Total: 72 names, Search batch: 10, LLM batch: 10
[PIPELINE] Using enhanced search, Dataset profile: high_connection


Processing search batches:   0%|          | 0/8 [00:00<?, ?batch/s]


[PIPELINE] ===== SEARCH BATCH 1/8 =====
[PIPELINE] Names: Akasha Gloria Hull, Akira Suzuki, Albert Warner Overhauser, Andrew J. Majda, A. Dale Kaiser, Ben R. Mottelson, Bob Hicok, Carl R. de Boor, Charles J. Arntzen, Chintamani Nagesa Ramachandra Rao
[SEARCH] Running parallel searches for 10 names (max 15 concurrent)...
[PROGRESS] Starting search for: Akasha Gloria Hull
[PROGRESS] Name: 'Akasha Gloria Hull' | Query: 'Akasha Gloria Hull "Purdue University"'
[PROGRESS] Starting search for: Akira Suzuki
[PROGRESS] Name: 'Akira Suzuki' | Query: 'Akira Suzuki "Purdue University"'
[PROGRESS] Starting search for: Albert Warner Overhauser
[PROGRESS] Name: 'Albert Warner Overhauser' | Query: 'Albert Warner Overhauser "Purdue University"'
[PROGRESS] Starting search for: Andrew J. Majda
[PROGRESS] Name: 'Andrew J. Majda' | Query: 'Andrew J. Majda "Purdue University"'
[PROGRESS] Starting search for: A. Dale Kaiser
[PROGRESS] Name: 'A. Dale Kaiser' | Query: 'A. Dale Kaiser "Purdue University"'
[PR

## 5. Results & Analysis

In [5]:
# Map results back to clusters
results_map = {name: res for name, res in zip(names_to_check, results)}

# Add results to dataframe
df_clustered['verdict'] = df_clustered['cluster_label'].map(lambda x: results_map.get(cluster_reps.set_index('cluster_label').loc[x, 'representative_name'], {}).get('verdict'))
df_clustered['institution'] = df_clustered['cluster_label'].map(lambda x: results_map.get(cluster_reps.set_index('cluster_label').loc[x, 'representative_name'], {}).get('institution'))
df_clustered['confidence'] = df_clustered['cluster_label'].map(lambda x: results_map.get(cluster_reps.set_index('cluster_label').loc[x, 'representative_name'], {}).get('confidence'))
df_clustered['evidence'] = df_clustered['cluster_label'].map(lambda x: results_map.get(cluster_reps.set_index('cluster_label').loc[x, 'representative_name'], {}).get('summary'))

# Identify Purdue Connections
def is_purdue_connected(row):
    inst = str(row.get('institution', '')).lower()
    return row['verdict'] == 'connected' and ('purdue' in inst)

df_clustered['is_connected'] = df_clustered.apply(is_purdue_connected, axis=1)

# Display Summary
connected_df = df_clustered[df_clustered['is_connected'] == True]

print(f"🏆 RESULTS SUMMARY")
print("="*60)
print(f"Total Analyzed: {len(df_clustered)}")
print(f"Confirmed Connections: {len(connected_df)}")
print(f"Connection Rate: {(len(connected_df)/len(df_clustered))*100:.1f}%")

print("\n🔍 CONFIRMED CONNECTIONS:")
display(connected_df[['name', 'verdict', 'confidence', 'evidence']])

# Export
df_clustered.to_excel("demo_results_100.xlsx", index=False)
print("\n💾 Results exported to 'demo_results_100.xlsx'")

🏆 RESULTS SUMMARY
Total Analyzed: 76
Confirmed Connections: 40
Connection Rate: 52.6%

🔍 CONFIRMED CONNECTIONS:


,name,verdict,confidence,evidence
0,Akira Suzuki,connected,medium,Akira Suzuki was a postdoctoral fellow at Purd...
1,Ben R. Mottelson,connected,high,Ben R. Mottelson earned a Bachelor of Science ...
2,E. M. Purcell,connected,medium,E. M. Purcell earned a Bachelor of Science in ...
4,Herbert C. Brown,connected,high,Herbert C. Brown was a professor at Purdue Uni...
6,Julian Schwinger,connected,medium,Julian Schwinger served as an instructor (facu...
7,Vernon L. Smith,connected,high,Vernon L. Smith served as a full professor at ...
8,Leah H. Jamieson,connected,high,Leah H. Jamieson currently holds a faculty pos...
10,Daniel L. Hartl,connected,high,Daniel L. Hartl previously served as a faculty...
11,Michael Friedman,connected,high,Michael (Elliot) Friedman is a current profess...
15,Ronald R. Breaker,connected,high,Ronald R. Breaker earned his Ph.D. in Biochemi...



💾 Results exported to 'demo_results_100.xlsx'


## 6. Appendix: Detailed Process Walkthrough (Example)
To understand the "black box" of the verification pipeline, here is a step-by-step execution for a single known connection (**Herbert C. Brown**).

In [6]:
import json
from institution_checker.main import process_name_search, process_name_llm
from institution_checker.search import _build_strategies
from institution_checker.llm_processor import _build_prompt

# Select a clear True Positive example
demo_name = "Herbert C. Brown"
institution = "Purdue University"

print(f"🔬 DETAILED WALKTHROUGH: {demo_name}")
print("="*60)

# 1. Query Generation
print(f"\n1️⃣  GENERATED SEARCH QUERIES:")
# We simulate the strategy generation to show what queries are used
strategies = _build_strategies(demo_name, institution, 5)
for s in strategies:
    print(f"   - [{s.name}]: {s.query}")

# 2. Search Execution
print(f"\n2️⃣  EXECUTING SEARCH (Live)...")
# We use the internal function to get raw results
_, search_results = await process_name_search(demo_name, use_enhanced_search=True)

print(f"   Found {len(search_results)} results. Top 3 examples:")
for i, res in enumerate(search_results[:3]):
    print(f"   {i+1}. {res['title']}")
    print(f"      {res['url']}")
    print(f"      Snippet: {res['snippet'][:120]}...")

# 3. LLM Prompt Construction
print(f"\n3️⃣  CONSTRUCTING LLM PROMPT:")
prompt = _build_prompt(demo_name, institution, search_results)
print(f"   [System]: You are an evidence-driven verifier...")
print(f"   [User]: ...Search findings:")
# Show a slice of the findings part of the prompt
start_marker = "Search findings:"
findings_start = prompt.find(start_marker)
if findings_start != -1:
    print(f"   {prompt[findings_start:findings_start+500]}...")
    print("   (...remaining results...)")

# 4. LLM Analysis
print(f"\n4️⃣  LLM DECISION:")
decision = await process_name_llm(demo_name, search_results)
print(json.dumps(decision, indent=2))

🔬 DETAILED WALKTHROUGH: Herbert C. Brown

1️⃣  GENERATED SEARCH QUERIES:
   - [name_institution_combined]: "Herbert C. Brown" "Purdue University"
   - [explicit_connection]: "Herbert C. Brown" "Purdue University" ("professor at" OR "faculty" OR "graduated from" OR "degree from" OR "alumnus" OR "worked at" OR "postdoc" OR "post-doc")
   - [education]: "Herbert C. Brown" "Purdue University" ("degree" OR "graduated" OR "alumni" OR "PhD" OR "bachelor")
   - [directory]: "Herbert C. Brown" "Purdue University" site:purdue.edu ("faculty" OR "staff" OR "directory")

2️⃣  EXECUTING SEARCH (Live)...
[PROGRESS] Starting search for: Herbert C. Brown
[PROGRESS] Name: 'Herbert C. Brown' | Query: 'Herbert C. Brown "Purdue University"'
[PROGRESS] Result sample: Herbert C. Brown: 1979 Nobel Prize in Chemistry
[PROGRESS] Basic search returned 20 results with max_score=24, sufficient for analysis
[PROGRESS] Search completed for Herbert C. Brown in 4.8s, found 20 results
   Found 20 results. Top 3 example

## 7. Simplified Usage
While the appendix above shows the internal mechanics, the actual usage for a single name is just one line of code.

In [ ]:

import institution_checker

# Configure the target institution
config.INSTITUTION = "Purdue University"

# Find connection for Name and Institution
result = await process_name("Herbert C. Brown", use_enhanced_search=True)

print(f"Name: {result['name']}")
print(f"Verdict: {result['verdict']}")
print(f"Evidence: {result['summary']}")

[PROGRESS] Starting search for: Herbert C. Brown
[PROGRESS] Name: 'Herbert C. Brown' | Query: 'Herbert C. Brown "Purdue University"'
[PROGRESS] Result sample: Herbert C. Brown: 1979 Nobel Prize in Chemistry
[PROGRESS] Basic search returned 20 results with max_score=24, sufficient for analysis
[PROGRESS] Search completed for Herbert C. Brown in 2.0s, found 20 results
[PROGRESS] Starting LLM analysis for: Herbert C. Brown
[PROGRESS] LLM analysis completed for Herbert C. Brown in 3.3s
Name: Herbert C. Brown
Verdict: connected
Evidence: Herbert C. Brown was a professor at Purdue University (affiliated during his 1979 Nobel Prize), a past faculty relationship.
